## TIFON Databases. Read and load example databases

In [1]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=0, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=1, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )
    
    return db

0 Warning! Import - NVTX not present!
/home/m.jaraiz/miniconda/envs/envkan_amd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Base de datos original
case_idx = list(range(100))
fuera = [64, 79, 87, 88, 94]
for c in fuera:
    case_idx.remove(c)
db_0 = read_db(datafolder='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3/', case_idx=case_idx)


 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3
100 simulations found.
Parse taked: 0.1256 seconds


In [3]:
# Base de datos adicional
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba/', case_idx='all')


 NEW CODA SIMULATION WILL BE LOADED FROM /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba
5 simulations found.
Parse taked: 0.0064 seconds


In [4]:
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])
flcc = db_1.data_dict['CADGroup_3']['FlCc'][:, :-1]
db_1.data_dict['CADGroup_3']['FlCc'] = flcc

['aoa', 'mach']
['aoa', 'mach', 'h']


### Merge two sets

In [5]:
db_completo = FRODO.merge_datasets(
    sources = [(db_0, '3'), (db_1, '3')],
    new_group_id='3_completo',
    k=4,
    mesh_ref=0,
    cache=True,
    get_df_metrics_attr={
        'var_metrics': ['CoefLift', 'CoefDrag', 'CoefMomentY'],
        'iter_var': 1000,
        'save' : False
    }
    
)
db_completo.summary_data()

Design vars: ['aoa', 'mach']
Design vars: ['aoa', 'mach', 'h']


AssertionError: get_df_metrics_attr provided but length of df_post does not match number of cases in FlCc. Please check that get_df_metrics_attr is correctly configured to extract a dataframe with the same number of rows as cases in FlCc for each dataset.

### Interpolation between meshes

In [ ]:
new_mesh = {
    "Coord": db_1.data_dict['CADGroup_3']['Coord'],
    "Conec": db_1.data_dict['CADGroup_3']['Conec'],
    "NodeCoord": db_1.data_dict['CADGroup_3']['NodeCoord'],
    # lo que tengas
}
db_0.sets.create_group_by_interpolation(
    id_group_src='3',
    new_group_id='3_interp',
    new_mesh=new_mesh,
    method='idw',
    k=8
    )

In [ ]:
db_0.summary_data()